# 27. Stat lines: goals, assists and goalie records in the simulator

Notebook 26's simulator plays the 2026-27 season and reports team points and playoff odds. This notebook gives every simulated season **player stat lines**:

- **skaters:** games, goals, assists, points;
- **goalies:** games, wins, losses, OT losses, goals-against average.

**How** (code: `src/scoring.py`, used by `src/season.py`):

1. **Scoring rates.** Each skater's goals per 60 and assists per 60, projected like notebook 15 projects WAR: his last 3 seasons, pulled toward his position's average, adjusted for age. Rookies and prospects get NHLe-based rates.
2. **His random season.** The same notebook 17 draw that moves his WAR also moves his scoring rates.
3. **Goals.** A team's simulated goals go to its dressed skaters by goals/60 x minutes.
4. **Assists.** Each goal gets 0, 1 or 2 assists at the real mix, by assists/60 x minutes, never to the scorer.
5. **Goalies.** Each game's decision and goals against go to the goalie who started it (notebook 26's back-to-back rule).

**After opening night,** a player's line is **real stats so far** (from the NHL's boxscores) **plus the simulated rest**.

**Save % is out for now:** the simulator has no shots.

## 1. Setup

In [1]:
import sys
sys.path.append("../src")

import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

from forecast import history_table, backtest, rmse, add_age, GOALS_PER_WIN
from edge_data import (build_bio_table, season_schedule, download_boxscores, boxscore_lines, download_club_stats,
                       club_stats_table)
from nhle import load_profiles
from development import training_nhle, best_nhle, experience, TARGETS
import lineups as L
import simulate as S
import scoring as SC
import season as SE

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
REFRESH = False          # True: download the club stats again (32 NHL calls). New boxscores are always fetched.
BUNDLE_DIR = DATA / "simulator_2026-27"
TRADES_PATH = DATA / "trades_2026-27.csv"

def label(s):
    return f"{s}-{str(s + 1)[-2:]}"

TRAIN_END = 2017
TRAIN = [t for t in TARGETS if t <= TRAIN_END]
TEST = [t for t in TARGETS if t > TRAIN_END]
RECENT = [2023, 2024, 2025]
REGS = [250, 500, 1000, 1500, 3000]           # regression minutes tried
MIN_TOI = 300                                 # backtest: players with 300+ minutes in the target season
N_RUN, SEED = 10000, 27

skater_war = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalie_war = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
bios = build_bio_table(DATA / "bio_raw", 2026)

## 2. Scoring history and the assist mix

Every skater-season from MoneyPuck, all situations: goals, primary and secondary assists, and minutes.

**The assist mix:** every goal with a secondary assist also has a primary one. So from the league's totals:

- goals with no assist = 1 - primary assists / goals;
- with two = secondary assists / goals;
- with one = the rest.

Measured on 2021-22 to 2025-26.

In [2]:
raw = SC.scoring_history([DATA / "skaters_2008_to_2024.csv", DATA / "skaters25-26.csv"], skater_war)
assert not raw.duplicated(["playerId", "season"]).any()
per = raw.groupby(["season", "group"]).apply(lambda d: pd.Series({"goals/60": d["goals"].sum() / d["toi"].sum() * 60,
                                                                   "assists/60": d["assists"].sum() / d["toi"].sum() * 60}),
                                             include_groups=False).unstack("group")
per.index = per.index.map(label)
print(f"{len(raw):,} skater-seasons, {raw['season'].min()}-{raw['season'].max()}. League rates per 60 (all situations):")
print(per.round(3).to_string())
MIX = SC.assist_mix(raw, [2021, 2022, 2023, 2024, 2025])
print(f"\nAssist mix 2021-22 to 2025-26: no assist {MIX['p0']:.1%}, one {MIX['p1']:.1%}, two {MIX['p2']:.1%} "
      f"({MIX['p1'] + 2 * MIX['p2']:.2f} assists per goal)")

16,271 skater-seasons, 2008-2025. League rates per 60 (all situations):
        goals/60        assists/60       
group          D      F          D      F
season                                   
2008-09    0.210  0.829      0.748  1.166
2009-10    0.200  0.805      0.734  1.120
2010-11    0.192  0.795      0.711  1.100
2011-12    0.189  0.770      0.705  1.061
2012-13    0.206  0.757      0.685  1.079
2013-14    0.216  0.758      0.719  1.057
2014-15    0.217  0.750      0.719  1.055
2015-16    0.215  0.755      0.712  1.030
2016-17    0.213  0.774      0.732  1.051
2017-18    0.233  0.828      0.777  1.143
2018-19    0.228  0.847      0.783  1.144
2019-20    0.233  0.843      0.792  1.138
2020-21    0.211  0.828      0.779  1.117
2021-22    0.229  0.887      0.853  1.196
2022-23    0.237  0.892      0.842  1.217
2023-24    0.227  0.879      0.849  1.178
2024-25    0.223  0.855      0.808  1.160
2025-26    0.240  0.867      0.837  1.186

Assist mix 2021-22 to 2025-26: no assist 6.5%

## 3. Projected scoring rates: the backtest

**The projection,** like notebook 15 and notebook 18, separately for goals and assists:

- the last 3 seasons, weighted equally;
- pulled toward the position's average with R minutes of weight;
- **age adjustment** off or on. On is notebook 18's adjust-then-blend, with each stat's own aging curve, refit without the target season.

**The backtest:** notebook 18's split. Players with 300+ minutes in the target season and a season the year before. The error is the minutes-weighted RMSE per 60.

**Compared with:** last season alone, and the position average.

**The pick:** lowest test error, with the training check.

In [3]:
TABLES = {stat: SC.rate_table(raw, stat) for stat in ["goals", "assists"]}
BT, rows = {}, []
start = time.time()
for stat, tab in TABLES.items():
    for aging in [False, True]:
        version = "adjust_then_blend" if aging else "none"
        for reg in REGS:
            b = backtest(tab, TARGETS, SC.WEIGHTS, reg, MIN_TOI, bios, aging={"F": version, "D": version, "G": "none"})
            BT[(stat, aging, reg)] = b
            for period, ts in [("train", TRAIN), ("test", TEST)]:
                rows.append({"stat": stat, "model": f"3 seasons, age {'on' if aging else 'off'}", "R": reg, "period": period,
                             "error": rmse(b[b["season"].isin(ts)])})
    for name, kw in [("last season alone", {"weights": (1, 0, 0), "reg": 0, "avg": False}),
                     ("position average", {"weights": SC.WEIGHTS, "reg": 1000, "avg": True})]:
        b = backtest(tab, TARGETS, kw["weights"], kw["reg"], MIN_TOI, bios, average_only=kw["avg"])
        BT[(stat, name)] = b
        for period, ts in [("train", TRAIN), ("test", TEST)]:
            rows.append({"stat": stat, "model": name, "R": "-", "period": period, "error": rmse(b[b["season"].isin(ts)])})
GRID = pd.DataFrame(rows).pivot_table(index=["stat", "model", "R"], columns="period", values="error", sort=False)[["train", "test"]]
print(f"Backtest error (per 60, minutes-weighted), {time.time() - start:.0f} seconds:")
print(GRID.round(4).to_string())

PICK = {}
for stat in TABLES:
    g = GRID.loc[stat].drop(index=["last season alone", "position average"], level="model")
    best_test, best_train = g["test"].idxmin(), g["train"].idxmin()
    PICK[stat] = {"aging": best_test[0].endswith("on"), "reg": int(best_test[1])}
    print(f"\n{stat}: pick {best_test[0]}, R = {best_test[1]} (test {g['test'].min():.4f}); best on training: "
          f"{best_train[0]}, R = {best_train[1]} ({'agrees' if best_train == best_test else 'DIFFERS'})")

bg, ba = BT[("goals", PICK["goals"]["aging"], PICK["goals"]["reg"])], BT[("assists", PICK["assists"]["aging"], PICK["assists"]["reg"])]
pts = bg.assign(pred=bg["pred"] + ba["pred"].to_numpy(), actual=bg["actual"] + ba["actual"].to_numpy())
assert (bg["playerId"].to_numpy() == ba["playerId"].to_numpy()).all()
for name in ["last season alone", "position average"]:
    a, b = BT[("goals", name)], BT[("assists", name)]
    x = a.assign(pred=a["pred"] + b["pred"].to_numpy(), actual=a["actual"] + b["actual"].to_numpy())
    print(f"Points/60, {name}: train {rmse(x[x['season'].isin(TRAIN)]):.4f}, test {rmse(x[x['season'].isin(TEST)]):.4f}")
print(f"Points/60, the pick: train {rmse(pts[pts['season'].isin(TRAIN)]):.4f}, test {rmse(pts[pts['season'].isin(TEST)]):.4f}")

Backtest error (per 60, minutes-weighted), 13 seconds:
period                            train    test
stat    model              R                   
goals   3 seasons, age off 250   0.2313  0.2440
                           500   0.2260  0.2417
                           1000  0.2237  0.2420
                           1500  0.2247  0.2445
                           3000  0.2313  0.2538
        3 seasons, age on  250   0.2283  0.2405
                           500   0.2230  0.2383
                           1000  0.2206  0.2388
                           1500  0.2216  0.2414
                           3000  0.2286  0.2512
        last season alone  -     0.3114  0.3494
        position average   -     0.2789  0.3169
assists 3 seasons, age off 250   0.3141  0.3272
                           500   0.3096  0.3260
                           1000  0.3103  0.3305
                           1500  0.3145  0.3378
                           3000  0.3296  0.3603
        3 seasons, age on  250   

## 4. Rookies and prospects: NHLe

A player without NHL seasons gets his rate from **NHLe** (notebook 16: his pre-NHL scoring turned into NHL points per game):

- **points/60** = NHLe x 60 / a rookie's minutes per game (the median for his position, notebook 23);
- **split into goals and assists** by his position's goal share (the 3 seasons before).

**How good is that?** Every past rookie with 20+ NHL games in a target season is checked. His NHLe comes from his seasons before, with notebook 16's method fit through 2017 only. That's compared with "every rookie gets the rookie average for his position", and with a **shrink**: average + b x (NHLe rate - average). b is fit on the training rookies.

In [4]:
history = history_table(skater_war, goalie_war)
profiles = load_profiles(DATA / "bio_raw")
careers = pd.read_csv(DATA / "career_totals.csv")
exp = experience(history, profiles, history[["playerId", "season"]])
usage = L.usage_table(skater_war, goalie_war, L.faceoff_rates([DATA / "skaters_2008_to_2024.csv", DATA / "skaters25-26.csv"]))
f_train, m_train = training_nhle(careers, TRAIN_END)
NH = best_nhle(careers, f_train, m_train, rule="last2")
r = raw.merge(exp, on=["playerId", "season"], how="left")

def rookie_context(seasons):
    """Rookie minutes per game, rookie average points/60 and the goal share of points, by group, in `seasons`."""
    p = r[r["season"].isin(seasons)]
    rk = p[(p["experience"] == 0) & (p["gp"] >= 20)]
    return (L.rookie_defaults(usage, exp, seasons),
            rk.groupby("group").apply(lambda d: d["points"].sum() / d["toi"].sum() * 60, include_groups=False),
            p.groupby("group").apply(lambda d: d["goals"].sum() / d["points"].sum(), include_groups=False))

rows = []
for t in TARGETS:
    mins, avg, share = rookie_context([t - 3, t - 2, t - 1])
    x = r[(r["season"] == t) & (r["experience"] == 0) & (r["gp"] >= 20)].merge(NH, on="playerId", how="left")
    x = x[x["nhle_season"] < t]                                   # NHLe from before his rookie season
    x["nhle_p60"] = x["nhle_ppg"] * 60 / x["group"].map(mins)
    x["avg_p60"] = x["group"].map(avg)
    rows.append(x)
RK = pd.concat(rows, ignore_index=True)
RK["actual"] = RK["points"] / RK["toi"] * 60
err = lambda p, d: float(np.sqrt(np.average((p - d["actual"]) ** 2, weights=d["toi"])))
tr, te = RK[RK["season"].isin(TRAIN)], RK[RK["season"].isin(TEST)]
shrinks = np.round(np.arange(0, 1.0001, 0.05), 2)
SHRINK = float(shrinks[int(np.argmin([err(tr["avg_p60"] + b * (tr["nhle_p60"] - tr["avg_p60"]), tr) for b in shrinks]))])
out = []
for name, f in [("rookie average", lambda d: d["avg_p60"]), ("NHLe as is", lambda d: d["nhle_p60"]),
                (f"NHLe shrunk (b = {SHRINK})", lambda d: d["avg_p60"] + SHRINK * (d["nhle_p60"] - d["avg_p60"]))]:
    out.append({"rookie rate": name, "train error": err(f(tr), tr), "test error": err(f(te), te),
                "train corr": f(tr).corr(tr["actual"]), "test corr": f(te).corr(te["actual"])})
print(f"Past rookies (20+ games): {len(tr)} training, {len(te)} test. Points/60, minutes-weighted error:")
print(pd.DataFrame(out).round(3).to_string(index=False))
print("\nAverage points/60 by position (all rookies): NHLe as is vs. actual")
print(RK.groupby("group")[["nhle_p60", "actual", "avg_p60"]].mean().round(3).to_string())

Past rookies (20+ games): 728 training, 481 test. Points/60, minutes-weighted error:
          rookie rate  train error  test error  train corr  test corr
       rookie average        0.554       0.502       0.441      0.532
           NHLe as is        0.566       0.610       0.595      0.604
NHLe shrunk (b = 0.5)        0.474       0.443       0.609      0.653

Average points/60 by position (all rookies): NHLe as is vs. actual
       nhle_p60  actual  avg_p60
group                           
D         0.861   0.838    0.887
F         1.666   1.466    1.622


## 5. 2026-27 rates for every player in the pool

- **NHL skaters** (notebook 18's source): the projection picked in section 3.
- **Prospects and new players** (notebooks 19 and 22): NHLe, shrunk as in section 4, with 2023-24 to 2025-26 rookie minutes, rookie average and goal shares. Without an NHLe, the rookie average.

In [5]:
B = SE.load_bundle(BUNDLE_DIR, TRADES_PATH)
pool = B["players"][B["players"]["in_pool"] & (B["players"]["group"] != "G")][["playerId", "name", "team", "group"]].copy()
ft = pd.read_csv(DATA / "forecast_table_2026-27.csv").set_index("playerId")
pool["source"] = pool["playerId"].map(ft["source"])
proj_g = SC.project_rates(TABLES["goals"], bios, 2026, PICK["goals"]["reg"], PICK["goals"]["aging"])
proj_a = SC.project_rates(TABLES["assists"], bios, 2026, PICK["assists"]["reg"], PICK["assists"]["aging"])
pool["g60"] = pool["playerId"].map(proj_g)
pool["a60"] = pool["playerId"].map(proj_a)
mins, avg, share = rookie_context(RECENT)
new = pool["g60"].isna()
rg, ra = SC.rookie_rates(pool.loc[new, "playerId"].map(ft["nhle_last2"]), pool.loc[new, "group"],
                         pool.loc[new, "group"].map(mins), pool.loc[new, "group"].map(avg), pool.loc[new, "group"].map(share), SHRINK)
pool.loc[new, "g60"], pool.loc[new, "a60"] = rg, ra
pool["rate_source"] = np.where(~new, "NHL projection", np.where(pool["playerId"].map(ft["nhle_last2"]).notna(), "NHLe", "rookie average"))
assert pool[["g60", "a60"]].notna().all().all() and (pool[["g60", "a60"]] >= 0).all().all()
print(pd.crosstab(pool["source"], pool["rate_source"]).to_string())
print(f"\nRookie context (2023-24 to 2025-26): minutes {mins.round(1).to_dict()}, points/60 {avg.round(3).to_dict()}, "
      f"goal share {share.round(3).to_dict()}")
top = pool.assign(p60=pool["g60"] + pool["a60"]).sort_values("p60", ascending=False).head(15)
last = raw[raw["season"] == 2025].set_index("playerId")
top["2025-26 points/60"] = top["playerId"].map(last["points"] / last["toi"] * 60)
print("\nHighest projected points/60:")
print(top[["name", "team", "g60", "a60", "p60", "2025-26 points/60", "rate_source"]].round(3).to_string(index=False))

rate_source             NHL projection  NHLe  rookie average
source                                                      
NHL skater (nb18)                  939     0               0
new to data (nb22)                   0     6               0
skater prospect (nb19)               0   263               1

Rookie context (2023-24 to 2025-26): minutes {'D': 16.0, 'F': 11.7, 'G': 3.0}, points/60 {'D': 0.951, 'F': 1.694}, goal share {'D': 0.217, 'F': 0.425}

Highest projected points/60:
             name team   g60   a60   p60  2025-26 points/60    rate_source
  Nikita Kucherov  TBL 1.396 3.000 4.397              5.047 NHL projection
   Connor McDavid  EDM 1.212 2.963 4.175              4.392 NHL projection
 Nathan MacKinnon  COL 1.405 2.539 3.944              4.279 NHL projection
   Leon Draisaitl  EDM 1.550 2.192 3.742              4.150 NHL projection
   David Pastrnak  BOS 1.381 2.274 3.654              3.772 NHL projection
Macklin Celebrini  SJS 1.356 2.052 3.409              3.948 NHL

## 6. Linking scoring to each player's random season

In each simulated season, a skater's rates are multiplied by

`m = exp(k x z) / (the average of exp(k x z) over his draw pool)`

- **z** is his notebook 17 draw, the same number that moves his WAR that season. A breakout year in WAR is also a breakout year in scoring.
- **Dividing by the pool average** keeps the average m at 1, so the projection stays the center.

**Fitting k on past seasons:** for every player-season in the backtest (projection from the seasons before, his **real** minutes), simulate goals and assists 1,000 times as Poisson counts at projected rate x m x minutes. k is chosen on the **training seasons** so 80% of real points land inside the simulated 80% range. The test seasons check it.

**k = 0** is game-to-game luck only: no talent changes.

In [6]:
INPUTS = S.draw_inputs(pd.read_csv(DATA / "dev_residuals.csv"), pd.read_csv(DATA / "dev_ranges.csv"), center_pools=True)
d = bg[["season", "playerId", "group", "toi", "pred", "actual"]].rename(columns={"pred": "g60", "actual": "g60_real"})
d["a60"], d["a60_real"] = ba["pred"].to_numpy(), ba["actual"].to_numpy()
d["G"] = np.round(d["g60_real"] * d["toi"] / 60)
d["A"] = np.round(d["a60_real"] * d["toi"] / 60)
d["P"] = d["G"] + d["A"]
d["age"] = add_age(d[["playerId", "season"]], bios)["age"].to_numpy()
d["draft_overall"] = d["playerId"].map(bios.set_index("playerId")["draft_overall"])
keys = sorted(INPUTS["pools"])
key_id = {k_: i for i, k_ in enumerate(keys)}
d["pool_id"] = [key_id[b_] for b_ in S.draw_buckets(d["group"], d["age"], d["draft_overall"], INPUTS)]
POOLS = [INPUTS["pools"][k_] for k_ in keys]
N_LINK = 1000
rng = np.random.default_rng(27)
Z = np.zeros((len(d), N_LINK))
for i, (z_, p_) in enumerate(POOLS):
    idx = np.flatnonzero(d["pool_id"].to_numpy() == i)
    Z[idx] = rng.choice(z_, size=(len(idx), N_LINK), p=p_)

def coverage(k):
    """Share of real G, A, P inside the simulated 80% and 50% ranges, training and test seasons."""
    m = SC.scoring_multiplier(Z, d["pool_id"].to_numpy(), SC.multiplier_norms(POOLS, k), k)
    rng2 = np.random.default_rng(28)
    G = rng2.poisson((d["g60"] * d["toi"] / 60).to_numpy()[:, None] * m)
    A = rng2.poisson((d["a60"] * d["toi"] / 60).to_numpy()[:, None] * m)
    out = []
    for period, ts in [("train", TRAIN), ("test", TEST)]:
        s_ = d["season"].isin(ts).to_numpy()
        row = {"k": k, "period": period}
        for name, sim in [("G", G), ("A", A), ("P", G + A)]:
            real_ = d.loc[s_, name].to_numpy()
            q = np.percentile(sim[s_], [10, 25, 75, 90], axis=1)
            row[f"{name} 80%"] = ((real_ >= q[0]) & (real_ <= q[3])).mean()
            row[f"{name} 50%"] = ((real_ >= q[1]) & (real_ <= q[2])).mean()
        out.append(row)
    return out

start = time.time()
LINK = pd.DataFrame([row for k in np.round(np.arange(0, 0.301, 0.02), 2) for row in coverage(k)])
tr_ = LINK[LINK["period"] == "train"].set_index("k")
K_LINK = float(round((tr_["P 80%"] - 0.8).abs().idxmin(), 2))
print(f"{len(d):,} player-seasons ({d['season'].isin(TRAIN).sum():,} training), {N_LINK:,} draws each, {time.time() - start:.0f} seconds")
print(LINK.round(3).to_string(index=False))
x = LINK[(LINK["k"] == K_LINK)].set_index("period")
print(f"\nk = {K_LINK} (fit on training seasons: points in the 80% range {x.loc['train', 'P 80%']:.1%}). "
      f"Test seasons: points 80% range {x.loc['test', 'P 80%']:.1%}, 50% range {x.loc['test', 'P 50%']:.1%}; "
      f"goals {x.loc['test', 'G 80%']:.1%}, assists {x.loc['test', 'A 80%']:.1%} in the 80% range")
k0 = LINK[(LINK["k"] == 0) & (LINK["period"] == "test")].iloc[0]
print(f"With k = 0 (game luck only), test: points 80% range {k0['P 80%']:.1%}: too narrow, as expected")

8,562 player-seasons (4,744 training), 1,000 draws each, 23 seconds
   k period  G 80%  G 50%  A 80%  A 50%  P 80%  P 50%
0.00  train  0.758  0.513  0.696  0.443  0.664  0.401
0.00   test  0.758  0.504  0.710  0.457  0.667  0.423
0.02  train  0.762  0.513  0.697  0.448  0.666  0.402
0.02   test  0.756  0.504  0.709  0.457  0.670  0.421
0.04  train  0.762  0.516  0.703  0.447  0.675  0.410
0.04   test  0.760  0.508  0.712  0.457  0.684  0.425
0.06  train  0.767  0.520  0.710  0.451  0.685  0.415
0.06   test  0.768  0.509  0.725  0.466  0.692  0.435
0.08  train  0.774  0.525  0.719  0.460  0.704  0.428
0.08   test  0.773  0.515  0.734  0.476  0.708  0.447
0.10  train  0.781  0.532  0.734  0.471  0.721  0.445
0.10   test  0.781  0.528  0.746  0.488  0.726  0.463
0.12  train  0.790  0.538  0.750  0.480  0.741  0.465
0.12   test  0.789  0.531  0.758  0.498  0.746  0.480
0.14  train  0.801  0.551  0.762  0.498  0.760  0.485
0.14   test  0.799  0.542  0.777  0.511  0.771  0.501
0.16  train  0

## 7. Giving out goals and assists: the whole season at once vs. goal by goal

**The simulator gives out a team's season of goals in one draw.** The scorers are weighted by goals/60 x m x minutes.

- **Goals:** lineups are fixed all season, so every goal has the same chances. Handing out N goals one at a time is *exactly* the same as one multinomial draw of N.
- **Assists:** the team's assists (0, 1 or 2 per goal, at the real mix) go out the same way, weighted by assists/60 x m x minutes.
  - **What about "never the scorer"?** A player's assists/60 is measured on real goals he *didn't* score, so the rule is already in his rate. Weighting by it gives each player exactly his projected share on average.

**The check** (Detroit's opening-night lineup, no draws, 2,000 seasons of about 250 goals): the season-level split vs. a goal-by-goal version, where the scorer can't assist and nobody gets both assists on one goal. Its assister weights are adjusted so averages match. **What to look for:**

- the averages should match;
- the goal-by-goal season totals spread a bit less, because a top playmaker can't pick up two assists on one goal.

**Why the season-level split stays:** section 6 fit k on real player-seasons with goals and assists counted the same way. So the spread it produces is the one calibrated to real seasons (82% of real point totals inside the 80% range on test seasons). With the goal-by-goal version, k would need refitting to land in the same place.

In [7]:
B = SE.load_bundle(BUNDLE_DIR, TRADES_PATH)
lu = SE.scenario_lineups(B)
det = lu[(lu["team"] == "DET") & lu["dressed"] & (lu["group"] != "G")].merge(pool[["playerId", "g60", "a60"]], on="playerId")
s_w = (det["g60"] * det["minutes_per_game"]).to_numpy()
s_w = s_w / s_w.sum()                        # each player's share of the team's goals
a_w = (det["a60"] * det["minutes_per_game"]).to_numpy()
a_w = a_w / a_w.sum()                        # ... and of its assists
N_CHECK = 2000
team_goals = np.random.default_rng(1).poisson(250, N_CHECK)
start = time.time()
fg, fa = SC.split_season(team_goals[:, None], s_w[None, None, :].repeat(N_CHECK, 0), a_w[None, None, :].repeat(N_CHECK, 0),
                         MIX, np.random.default_rng(2))
fg, fa = fg[:, 0, :], fa[:, 0, :]
t_fast = time.time() - start
start = time.time()
eg, ea = SC.split_by_goal(team_goals, s_w, SC.goal_by_goal_weights(s_w, a_w, MIX), MIX, np.random.default_rng(3))
t_exact = time.time() - start
assert (fg.sum(1) == team_goals).all() and (eg.sum(1) == team_goals).all()
cmp = pd.DataFrame({"player": det["name"], "G season": fg.mean(0), "G by goal": eg.mean(0), "A target": a_w * fa.sum(1).mean(),
                    "A season": fa.mean(0), "A by goal": ea.mean(0), "P sd season": (fg + fa).std(0), "P sd by goal": (eg + ea).std(0)})
cmp["sd ratio"] = cmp["P sd season"] / cmp["P sd by goal"]
print(f"Season-level split: {t_fast:.2f} s; goal by goal: {t_exact:.1f} s (one team, {N_CHECK:,} seasons)")
print(cmp.round(2).to_string(index=False))
print(f"\nAverages: the largest gap in goals is {(cmp['G season'] - cmp['G by goal']).abs().max():.2f}, in assists "
      f"{(cmp['A season'] - cmp['A by goal']).abs().max():.2f}. Team assists per goal: season-level {fa.sum(1).mean() / team_goals.mean():.3f}, "
      f"goal by goal {ea.sum(1).mean() / team_goals.mean():.3f}")
print(f"Spread of a player's season points: the season-level split is {cmp['sd ratio'].min() - 1:+.0%} to {cmp['sd ratio'].max() - 1:+.0%} "
      f"wider than goal by goal (widest for the top playmakers)")

Season-level split: 0.01 s; goal by goal: 6.0 s (one team, 2,000 seasons)
                  player  G season  G by goal  A target  A season  A by goal  P sd season  P sd by goal  sd ratio
            Dylan Larkin     32.36      32.13     37.66     37.71      37.50         9.21          8.29      1.11
           Lucas Raymond     28.28      28.23     49.46     49.04      49.42         9.67          8.77      1.10
          Alex DeBrincat     33.12      32.74     37.11     37.11      37.19         9.24          8.49      1.09
           Emmitt Finnie     15.17      15.10     20.84     20.97      20.93         6.51          6.04      1.08
             Andrew Copp     11.98      12.12     24.90     24.78      24.91         6.37          6.13      1.04
        Viktor Arvidsson     21.06      21.23     24.43     24.41      24.58         7.06          6.82      1.03
            J.T. Compher     12.16      11.96     20.22     20.30      20.33         5.78          5.68      1.02
            Ma

## 8. Real 2026-27 stats so far: boxscores, checked against the NHL's totals

A player's full-season line after opening night = his **real stats before the start date** + the **simulated rest**. The real stats come from each played game's boxscore:

- **The download:** one call per game, saved to `data/schedule_raw/2026-27/boxscores/`. A finished game never changes, so only games new since the last run are downloaded. Run notebook 26 with `REFRESH = True` first to pick up new games, then this notebook.
- **What counts:** a player gets a game when he gets ice time. Boxscores also list the backup goalie who dressed but didn't play.
- **The check:** the NHL's season totals per team (club stats, 32 calls, saved once; `REFRESH = True` reloads them) must match the boxscores for every player.

In [8]:
SCHED, STAMP = season_schedule(DATA / "schedule_raw" / "2026-27")
played = SCHED[SCHED["played"]]
failed = download_boxscores(played["game_id"].tolist(), DATA / "schedule_raw" / "2026-27" / "boxscores")
assert not failed
BOX = boxscore_lines(DATA / "schedule_raw" / "2026-27" / "boxscores")
BOX = BOX[BOX["game_id"].isin(played["game_id"])]
assert BOX["game_id"].nunique() == len(played)
REAL = BOX[BOX["toi"] > 0]
print(f"{len(played)} games through {played['date'].max()}: {len(REAL):,} player-games with ice time "
      f"({(BOX['toi'] == 0).sum()} dressed with none: backup goalies and {((BOX['toi'] == 0) & (BOX['group'] != 'G')).sum()} skater)")

teams = sorted(set(SCHED["home"]))
assert not download_club_stats(teams, DATA / "current_raw" / "club_stats", refresh=REFRESH)
cs_sk, cs_g = club_stats_table(DATA / "current_raw" / "club_stats")
mine = REAL[REAL["group"] != "G"].groupby(["team", "playerId"]).agg(gp=("game_id", "size"), goals=("goals", "sum"), assists=("assists", "sum"))
j = cs_sk.set_index(["team", "playerId"])[["gp", "goals", "assists"]].join(mine, rsuffix="_box", how="outer").fillna(0)
bad = j[(j["gp"] != j["gp_box"]) | (j["goals"] != j["goals_box"]) | (j["assists"] != j["assists_box"])]
g_ = REAL[REAL["group"] == "G"].assign(w=lambda x: x["decision"] == "W", l=lambda x: x["decision"] == "L", o=lambda x: x["decision"] == "O")
mine_g = g_.groupby(["team", "playerId"]).agg(gp=("game_id", "size"), wins=("w", "sum"), losses=("l", "sum"), otl=("o", "sum"),
                                               goals_against=("goals_against", "sum"), toi=("toi", "sum"))
jg = cs_g.set_index(["team", "playerId"])[["gp", "wins", "losses", "otl", "goals_against", "toi"]].join(mine_g, rsuffix="_box", how="outer").fillna(0)
bad_g = jg[(jg[["gp", "wins", "losses", "otl", "goals_against"]].to_numpy() != jg[["gp_box", "wins_box", "losses_box", "otl_box", "goals_against_box"]].to_numpy()).any(axis=1)
           | ((jg["toi"] - jg["toi_box"]).abs() > 0.02)]
print(f"Club stats vs. boxscores: {len(j)} skater rows, {len(bad)} mismatches; {len(jg)} goalie rows, {len(bad_g)} mismatches")
if len(bad) or len(bad_g):
    print(bad.head(10).to_string()); print(bad_g.head(10).to_string())
assert not len(bad) and not len(bad_g)
for name in ["Alex DeBrincat", "Dylan Larkin", "Lucas Raymond", "Connor McDavid"]:
    x = cs_sk[cs_sk["name"] == name].iloc[0]
    y = REAL[REAL["playerId"] == x["playerId"]]
    print(f"  {name:15s} NHL: {x['gp']} GP, {x['goals']} G, {x['assists']} A | boxscores: {len(y)} GP, {y['goals'].sum()} G, {y['assists'].sum()} A")
multi = REAL.groupby("playerId").agg(name=("name", "last"), teams=("team", lambda t: " then ".join(dict.fromkeys(t))),
                                     gp=("game_id", "size"), g=("goals", "sum"), a=("assists", "sum"))
multi = multi[multi["teams"].str.contains("then")]
print(f"\nPlayed for 2+ teams (their stats are combined by player): {len(multi)}")
print(multi.to_string())

Boxscores: 285 already saved, 0 downloaded
285 games through 2026-11-09: 10,859 player-games with ice time (541 dressed with none: backup goalies and 1 skater)
Club stats: 32 already saved, 0 downloaded
Club stats vs. boxscores: 765 skater rows, 0 mismatches; 76 goalie rows, 0 mismatches
  Alex DeBrincat  NHL: 18 GP, 7 G, 9 A | boxscores: 18 GP, 7 G, 9 A
  Dylan Larkin    NHL: 10 GP, 2 G, 4 A | boxscores: 10 GP, 2 G, 4 A
  Lucas Raymond   NHL: 18 GP, 6 G, 10 A | boxscores: 18 GP, 6 G, 10 A
  Connor McDavid  NHL: 19 GP, 11 G, 37 A | boxscores: 19 GP, 11 G, 37 A

Played for 2+ teams (their stats are combined by player): 8
                  name         teams  gp  g  a
playerId                                      
8480842   F. Hallander  PIT then EDM  11  1  2
8481030   J. St. Ivany  WPG then WSH   9  1  4
8481593     J. Struble  MTL then NSH  16  2  0
8482076        N. Daws  NJD then WPG   6  0  0
8482094        D. Hunt  MIN then NSH  18  0  1
8482110      D. Mercer  NJD then WPG  18  1

## 9. Saving the stat-line inputs to the bundle

Three new files in `data/simulator_2026-27/` (gitignored). Notebook 26's files are left as they are.

- **`scoring.csv`:** every pool skater's goals/60 and assists/60, and where they came from.
- **`scoring.json`:** the assist mix and k.
- **`real_games.csv`:** one row per player per real game so far, so the app can cut at any start date without calling the NHL.

In [9]:
pool[["playerId", "g60", "a60", "rate_source"]].rename(columns={"rate_source": "source"}).to_csv(BUNDLE_DIR / "scoring.csv", index=False)
(BUNDLE_DIR / "scoring.json").write_text(json.dumps({"k": K_LINK, "mix": MIX, "picks": PICK, "rookie_shrink": SHRINK,
                                                     "boxscores_through": str(played["date"].max())}, indent=1))
cols = ["game_id", "date", "team", "playerId", "name", "position", "group", "goals", "assists", "toi", "goals_against", "decision"]
BOX[cols].to_csv(BUNDLE_DIR / "real_games.csv", index=False)
B = SE.load_bundle(BUNDLE_DIR, TRADES_PATH)
assert "scoring" in B and "real_games" in B
print(f"Saved: {len(pool):,} skater rates, k = {K_LINK}, {len(BOX):,} real player-games (through {played['date'].max()})")

Saved: 1,209 skater rates, k = 0.18, 11,400 real player-games (through 2026-11-09)


## 10. Checks: standings unchanged, totals add up, same seed, speed

- **Stat lines use their own random numbers,** so turning them on must leave every team result exactly the same.
- **In every simulated season:**
  - **a team's goals by its skaters** = its simulated goals, minus the extra shootout "goal" the standings give the shootout winner. That's asserted inside the runner;
  - **goalies:** one decision per goalie per game.

In [10]:
plain = SE.run_scenario(B, n_sims=2000, seed=SEED)
stats = SE.run_scenario(B, n_sims=2000, seed=SEED, stat_lines=True, keep_seasons=True)
assert plain["table"].equals(stats["table"]) and np.array_equal(plain["points"], stats["points"])
again = SE.run_scenario(B, n_sims=2000, seed=SEED, stat_lines=True)
assert again["skaters"].equals(stats["skaters"]) and again["goalies"].equals(stats["goalies"])
gl = stats["goalies"]
assert np.isclose(gl["sim_wins"].sum() + gl["sim_losses"].sum() + gl["sim_otl"].sum(), 2 * len(SCHED))
assert np.isclose(gl["sim_wins"].sum(), len(SCHED))
print("Standings identical with and without stat lines: yes. Same seed, same stat lines: yes. "
      "One decision per goalie per game: yes")

# Goals by skaters = team goals minus shootout "goals", season by season (one season, every team)
one = SE.run_scenario(B, n_sims=1, seed=SEED, stat_lines=True, keep_seasons=True)
tg = pd.Series(one["season_goals"][0], index=one["skaters"]["team"]).groupby(level=0).sum()
print(f"One season: {int(tg.sum())} goals by skaters; average per team {tg.mean():.1f}")

for n in [1000, 10000]:
    res = SE.run_scenario(B, n_sims=n, seed=11, stat_lines=True)
    print(f"{n:,} seasons from opening night with stat lines: {res['seconds']:.1f} seconds")

Standings identical with and without stat lines: yes. Same seed, same stat lines: yes. One decision per goalie per game: yes
One season: 8295 goals by skaters; average per team 259.2
1,000 seasons from opening night with stat lines: 1.3 seconds
10,000 seasons from opening night with stat lines: 8.1 seconds


## 11. Goalies: records and goals-against average

- **Each game's decision** (W, L, OT loss) goes to the goalie who started it.
- **GAA** = goals against per 60 minutes in his starts. Empty-net goals (the leading team's late goals) and the shootout "goal" don't count. A start counts as 60 minutes.
- **Save % is left out:** the simulator has no shots.

**For comparison:** the real league GAA so far this season, from the boxscores.

In [11]:
BASE = SE.run_scenario(B, n_sims=N_RUN, seed=SEED, stat_lines=True, keep_seasons=True)
gl = BASE["goalies"].assign(gaa=lambda x: x["sim_ga"] / x["sim_gp"])
real_g = REAL[REAL["group"] == "G"]
print(f"League GAA: simulated {gl['sim_ga'].sum() / gl['sim_gp'].sum():.2f}; real 2026-27 so far "
      f"{real_g['goals_against'].sum() / real_g['toi'].sum() * 60:.2f}")
print("\nStarters, opening night (average of 10,000 seasons), best and worst GAA:")
st = gl[gl["role"] == "Starter"].sort_values("gaa")
show = pd.concat([st.head(6), st.tail(6)])
print(show[["name", "team", "sim_gp", "sim_wins", "sim_losses", "sim_otl", "gaa"]].round(2).to_string(index=False))

League GAA: simulated 2.87; real 2026-27 so far 2.98

Starters, opening night (average of 10,000 seasons), best and worst GAA:
                name team  sim_gp  sim_wins  sim_losses  sim_otl  gaa
      Logan Thompson  WSH      54     33.33       15.78     4.88 2.53
     Scott Wedgewood  COL      54     32.96       16.14     4.90 2.55
      Jake Oettinger  DAL      54     32.35       16.59     5.06 2.58
Ukko-Pekka Luukkonen  BUF      54     30.73       18.00     5.27 2.67
  Andrei Vasilevskiy  TBL      54     30.75       17.98     5.27 2.67
         Jakub Dobes  MTL      54     30.67       18.08     5.26 2.67
     Anthony Stolarz  TOR      54     25.35       22.77     5.89 2.96
       Darcy Kuemper  LAK      54     24.39       23.65     5.96 3.01
      Spencer Knight  CHI      54     24.34       23.69     5.98 3.01
        Joey Daccord  SEA      54     23.34       24.62     6.04 3.06
         Dustin Wolf  CGY      54     22.61       25.27     6.12 3.10
      Thatcher Demko  VAN      54

## 12. Realism: simulated 2026-27 seasons vs. real 2021-22 to 2025-26

**What's compared,** every simulated season against every real one:

- the league leader's points, goals and assists;
- how many players reach 100 points and 50 goals;
- the points of the 10th and 50th best scorers.

**Scaling:** the real seasons had 82 games, so each real total is scaled to 84.

**Said plainly:** simulated players are at **full health**. Every dressed skater plays every game, so the simulated seasons should have more big totals than real ones, where stars miss games.

In [12]:
P = BASE["season_goals"].astype(np.int32) + BASE["season_assists"]
G, A = BASE["season_goals"], BASE["season_assists"]
def season_marks(p, g, a):
    ps = -np.sort(-p, axis=1)
    return {"leader points": p.max(1), "leader goals": g.max(1), "leader assists": a.max(1),
            "100-point players": (p >= 100).sum(1), "50-goal scorers": (g >= 50).sum(1),
            "10th best points": ps[:, 9], "50th best points": ps[:, 49]}
sim = season_marks(P, G, A)
rows = []
for s_ in [2021, 2022, 2023, 2024, 2025]:
    x = raw[raw["season"] == s_]
    sc_ = 84 / 82
    marks = season_marks((x["points"].to_numpy() * sc_)[None, :], (x["goals"].to_numpy() * sc_)[None, :],
                         (x["assists"].to_numpy() * sc_)[None, :])
    rows.append(pd.Series({k: float(v[0]) for k, v in marks.items()}, name=label(s_)))
real_marks = pd.DataFrame(rows).T
real_marks["simulated median"] = [np.median(v) for v in sim.values()]
real_marks["simulated 10th"] = [np.percentile(v, 10) for v in sim.values()]
real_marks["simulated 90th"] = [np.percentile(v, 90) for v in sim.values()]
print("Real seasons (scaled to 84 games) vs. 10,000 simulated 2026-27 seasons from opening night:")
print(real_marks.round(1).to_string())

Real seasons (scaled to 84 games) vs. 10,000 simulated 2026-27 seasons from opening night:
                   2021-22  2022-23  2023-24  2024-25  2025-26  simulated median  simulated 10th  simulated 90th
leader points        126.0    156.7    147.5    124.0    141.4             133.0           117.0           158.0
leader goals          61.5     65.6     70.7     53.3     54.3              56.0            49.0            65.0
leader assists        87.1     91.2    102.4     86.0     92.2              89.0            78.0           107.0
100-point players      9.0     13.0     10.0      6.0      8.0               7.0             4.0            10.0
50-goal scorers        4.0      5.0      5.0      1.0      2.0               2.0             0.0             4.0
10th best points      99.4    104.5    100.4     93.2     98.3              95.0            90.9           101.0
50th best points      71.7     74.8     71.7     70.7     72.7              69.0            66.0            71.0


## 13. Hockey checks: the top projected scorers and Detroit's

For each player:

- **his rates** and **minutes per game** in the lineup;
- **his simulated season:** the average, and the 10th to 90th range;
- **his real 2025-26 line,** next to it.

In [13]:
sk = BASE["skaters"].merge(pool[["playerId", "g60", "a60"]], on="playerId")
sk["minutes"] = sk["playerId"].map(BASE["lineups"].set_index("playerId")["minutes_per_game"])
l25 = raw[raw["season"] == 2025].set_index("playerId")
for c in ["gp", "goals", "assists", "points"]:
    sk[f"2025-26 {c}"] = sk["playerId"].map(l25[c])
names = ["Connor McDavid", "Nathan MacKinnon", "Nikita Kucherov", "Leon Draisaitl", "Alex DeBrincat", "Lucas Raymond", "Dylan Larkin"]
cols = ["name", "team", "g60", "a60", "minutes", "sim_g", "sim_a", "sim_p", "sim_p_p10", "sim_p_p90",
        "2025-26 gp", "2025-26 goals", "2025-26 assists", "2025-26 points"]
print(sk.set_index("name").loc[names].reset_index()[cols].round(2).to_string(index=False))
print("\nTop 15 projected scorers (average of 10,000 seasons, opening night):")
print(sk.sort_values("sim_p", ascending=False).head(15)[cols].round(2).to_string(index=False))

            name team  g60  a60  minutes  sim_g  sim_a  sim_p  sim_p_p10  sim_p_p90  2025-26 gp  2025-26 goals  2025-26 assists  2025-26 points
  Connor McDavid  EDM 1.21 2.96    21.05  32.78  77.26 110.04       83.0      139.0        82.0           48.0             90.0           138.0
Nathan MacKinnon  COL 1.41 2.54    20.12  37.93  67.60 105.53       80.0      133.0        80.0           53.0             74.0           127.0
 Nikita Kucherov  TBL 1.40 3.00    19.09  35.69  69.44 105.13       80.0      133.0        76.0           44.0             86.0           130.0
  Leon Draisaitl  EDM 1.55 2.19    19.82  39.55  53.94  93.49       70.0      119.0        65.0           35.0             62.0            97.0
  Alex DeBrincat  DET 1.37 1.50    18.02  33.23  37.55  70.78       52.0       92.0        82.0           41.0             44.0            85.0
   Lucas Raymond  DET 1.15 1.95    18.45  28.70  50.23  78.93       58.0      102.0        80.0           25.0             51.0         

## 14. Reality mode: real so far + the simulated rest

**The setup:** from today, with every trade so far and Larkin and Edvinsson sitting (notebook 26's reality mode).

**Each line** is real stats before today + the simulated rest of the season.

- **Larkin** keeps his 10 real games but doesn't add any.
- **Traded players** carry their stats from both teams.

In [14]:
TODAY = STAMP[:10]
pid = lambda name: int(B["players"].loc[B["players"]["name"] == name, "playerId"].iloc[0])
NOW = SE.run_scenario(B, start_date=TODAY, sit_out=[pid("Dylan Larkin"), pid("Simon Edvinsson")], n_sims=N_RUN, seed=SEED, stat_lines=True)
s_lines, g_lines = SE.season_lines(NOW, B)
cols = ["name", "team", "real_gp", "real_g", "real_a", "real_p", "sim_gp", "sim_g", "sim_a", "sim_p", "gp", "g", "a", "p", "p_p10", "p_p90"]
print(f"From {TODAY}: real stats through {played['date'].max()}, then {NOW['games_simulated']:,} simulated games. Detroit:")
print(s_lines[s_lines["team"] == "DET"][cols].round(1).to_string(index=False))
print("\nDetroit's goalies:")
print(g_lines[g_lines["team"] == "DET"][["name", "role", "real_gp", "real_w", "real_l", "real_otl", "sim_gp", "gp", "w", "l", "otl", "gaa"]].round(2).to_string(index=False))
print("\nLeague's top 10 by season points (average):")
print(s_lines.head(10)[cols].round(1).to_string(index=False))
x = s_lines.set_index("playerId").loc[multi.index.intersection(s_lines["playerId"])]
print("\nTraded players, real stats combined across teams:")
print(x[["name", "team", "real_gp", "real_g", "real_a", "sim_gp", "gp", "p"]].round(1).to_string())

From 2026-11-10: real stats through 2026-11-09, then 1,059 simulated games. Detroit:
                    name team  real_gp  real_g  real_a  real_p  sim_gp  sim_g  sim_a  sim_p   gp    g    a    p  p_p10  p_p90
           Lucas Raymond  DET     18.0     6.0    10.0    16.0    66.0   24.3   41.6   65.9 84.0 30.3 51.6 81.9   64.0  102.0
          Alex DeBrincat  DET     18.0     7.0     9.0    16.0    66.0   28.0   30.9   59.0 84.0 35.0 39.9 75.0   58.0   93.0
        Viktor Arvidsson  DET     18.0     8.0     7.0    15.0    66.0   18.1   20.4   38.4 84.0 26.1 27.4 53.4   42.0   66.0
           Moritz Seider  DET     18.0     0.0    10.0    10.0    66.0    7.5   32.7   40.1 84.0  7.5 42.7 50.1   38.0   63.0
           Emmitt Finnie  DET     18.0     7.0     4.0    11.0    66.0   13.0   17.4   30.4 84.0 20.0 21.4 41.4   31.0   52.0
            Justin Faulk  DET     18.0     3.0     6.0     9.0    66.0    6.0   22.2   28.1 84.0  9.0 28.2 37.1   28.0   47.0
             Andrew Copp  DET    

## Takeaways
- **Scoring rates:** projected like notebook 15. The pick for both goals and assists is the last 3 seasons, age adjustment on, pulled toward the position average with **500 minutes** of weight.
  - **Goals/60, test error:** 0.238, vs. 0.349 for last season alone and 0.317 for the position average.
  - **Assists/60:** 0.322 (vs. 0.411 and 0.492).
  - **Points/60:** 0.432 (vs. 0.558 and 0.707).
  - **Said plainly:** for goals, the training seasons prefer 1,000 minutes (0.2206 vs. 0.2230). The difference is small, and the pick follows the test seasons.
- **Rookies and prospects (NHLe):** NHLe as is is *worse* than giving every rookie the position average (test error 0.610 vs. 0.502 points/60). It runs high, especially for forwards (1.67 projected vs. 1.47 real).
  - **Shrunk halfway** toward the rookie average (b = 0.5, fit on training rookies), it beats both: test error 0.443, correlation 0.65.
  - **In 2026-27:** 269 prospects and new players use it, and 1 (no NHLe) gets the rookie average.
- **The breakout link:** a skater's scoring rates move with his notebook 17 draw, m = exp(0.18 x z).
  - **k = 0.18 is fit on training seasons:** 80.2% of real point totals land in the simulated 80% range.
  - **Test seasons:** 81.8% of points (80% range), 53.6% (50% range); goals 82.6%, assists 80.7%.
  - **Without the link** (game luck only), only 66.7% of points land in the 80% range. Real player-seasons swing more than game luck alone explains.
- **Giving out goals and assists for the whole season at once** works out the same as goal by goal:
  - **Averages match:** within 0.4 goals and 0.4 assists per player for Detroit.
  - **The spread is up to 11% wider** for top playmakers, because goal by goal nobody gets both assists on one goal.
  - **Why the season-level split stays:** k was fit with goals and assists counted the same way, so its spread is the one calibrated to real seasons. It's also 600 times faster.
  - **"Never the scorer":** a player's assists/60 is measured on goals he didn't score, so weighting by it already gives him his real share.
- **The real mix:** 6.5% of goals have no assist, 18.4% one, 75.1% two (1.69 assists per goal).
- **Real stats so far:**
  - 285 boxscores (games through Nov 9). They match the NHL's club totals for all 765 skater rows and 76 goalie rows: DeBrincat 18 GP, 7 G, 9 A; Larkin 10 GP, 2 G, 4 A; McDavid 19 GP, 11 G, 37 A.
  - **A game counts only if he got ice time:** boxscores also list 540 backup goalies who dressed and didn't play, and 1 skater.
  - **8 players played for two teams;** their stats are combined.
- **Checks:**
  - standings are identical with stat lines on or off;
  - the same seed gives the same lines;
  - every team goal has a scorer (asserted in the runner);
  - one decision per goalie per game.
- **Speed:** 10,000 seasons with stat lines in 7.7 seconds (5.1 without), 1,000 in 1.1.
- **Goalies:** league GAA is 2.87 simulated vs. 2.98 real so far this season.
  - **Best starters:** Thompson (WSH), Wedgewood (COL), Oettinger (DAL), about 2.55.
  - **Worst:** Demko (VAN, 3.17), behind the weakest teams.
  - **No save % yet:** the simulator has no shots.
- **Realism vs. real 2021-22 to 2025-26** (scaled to 84 games; simulated median, with 10th-90th):

  | | Simulated | Real |
  |---|---|---|
  | League leader's points | 133 (117 to 158) | 124 to 157 |
  | League leader's goals | 56 | 53 to 71 |
  | League leader's assists | 89 | 86 to 102 |
  | 100-point players | 7 (4 to 10) | 6 to 13 |
  | 50-goal scorers | 2 | 1 to 5 |
  | 10th best scorer's points | 95 | 93 to 104 |
  | 50th best scorer's points | 69 (66 to 71) | 71 to 75 |

  - **Said plainly:** the top end looks like a real season. The 50th best scorer is a little low, because projections pull the second tier toward average and minutes are fixed.
  - **Full health** (everyone plays 84) works the other way, so the two partly cancel.
- **Hockey checks, opening night:**
  - **The top projected scorers:** McDavid 110 points (83 to 139), MacKinnon 106, Kucherov 105, Pastrnak 98, Draisaitl 93. All are below their 2025-26 totals (McDavid 138, MacKinnon 127, Kucherov 130), as projections pulled toward average are. Their 90th percentiles reach those seasons.
  - **Detroit:** Raymond 79 (real 76 last season), DeBrincat 71 (85), Larkin 71 (67).
  - **DeBrincat** projects lower because his assists/60 (1.50) is pulled toward the average.
- **Reality mode** (from Nov 10, Larkin and Edvinsson sitting): each line is real so far + the simulated rest.
  - Raymond 82 points (16 real + 66 simulated).
  - DeBrincat 75.
  - Larkin stays at his real 10 GP, 6 points.
  - Players sent down or traded away keep their real games. Traded players carry both teams' stats.
- **Files:** `scoring.csv`, `scoring.json` and `real_games.csv` are added to `data/simulator_2026-27/` (gitignored). The Forecast page's Simulator tab reads them.
- **To update with new games:** run notebook 26 with `REFRESH = True` (new schedule results), then this notebook (new boxscores download automatically; `REFRESH = True` here reloads the club totals for the check).